# Vision-Based Traversability Planning: A*, Temporal Fusion, and RAFT-Aligned Fusion

This notebook implements and evaluates image-space path planning on real semantic and geometric traversability cost maps derived from RGB driving sequences.

The experiments are designed to run in **Google Colab**, with input data stored in Google Drive. The notebook reads RGB frames and precomputed cost maps from `/content/drive/MyDrive/VisionBasedPlanning/datasets/scenario_XX/`, where `MyDrive` refers to the user's Google Drive directory mounted in Colab. Results are saved to `final_evaluation/scenario_XX/`.

The planner uses obstacle inflation, safe goal selection, cost-aware A* search, collision-aware path smoothing, and driving corridor visualization. The same planner is used across three configurations:

- **Method A — Independent A\*:** A* planning on the current combined cost map.
- **Method B — Temporal Fusion + A\*:** Fusion of the current and previous cost maps without motion alignment, followed by A* planning.
- **Method C — RAFT-Aligned Temporal Fusion + A\*:** RAFT optical flow is used to warp the previous cost map into the current frame before temporal fusion and A* planning.

The evaluation compares path stability, path length, obstacle clearance, and planning success across these methods. A separate experiment compares **semantic-only, geometric-only, and combined cost maps** to investigate the contribution of each traversability representation.

For each scenario, the notebook generates comparison and pipeline visualization videos, evaluation metrics, planned paths, and statistical analyses.

**Execution order:** Setup and configuration → helper functions → validation (inspect the diagnostic report and figures) → full evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import io, re, json, math, time, heapq, shutil, zipfile, subprocess
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
from scipy.ndimage import binary_dilation, distance_transform_edt, gaussian_filter1d
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights
from tqdm.auto import tqdm
from IPython.display import display

ROOT = Path('/content/drive/MyDrive/VisionBasedPlanning')
CACHE = Path('/content/cache')       # ZIPs are copied here once; reading from Drive per frame is slow

RUN_MODE = 'all'                  # 'single' or 'all'
RUN_MAIN_EXPERIMENT = True           # A vs B vs C (current map, temporal fusion, RAFT fusion)
RUN_SOURCE_EXPERIMENT = True         # semantic-only vs geometric-only vs combined costs, with and without fusion
SELECTED_SCENARIO = 'scenario_01'
MAX_FRAMES = None                      # None = all frames
PLAN_SIZE = (512, 272)               # (width, height); full resolution: (1024, 544)

# Cost combination
SEMANTIC_WEIGHT = 0.5
GEOMETRY_WEIGHT = 0.5
OBSTACLE_THRESHOLD = 0.75
HARD_OBSTACLES = True                # pixel is an obstacle if EITHER map alone is >= OBSTACLE_THRESHOLD

# Temporal fusion (identical for B and C; C additionally aligns with RAFT)
FUSION_ALPHA = 0.65                  # weight of the current map
FUSION_HISTORY = 'fused'             # 'fused' = recursive (previous fused map), 'raw' = previous combined map only
PRESERVE_CURRENT_OBSTACLES = False   # True = obstacles seen in the current frame can never be averaged away
RESET_ON_GAP = True                  # restart temporal history if frame IDs are not consecutive

# RAFT
RAFT_ITERS = 12
FB_CHECK = True                      # forward-backward consistency check to reject unreliable flow

# Planner parameters, defined at the original notebook's 512-px width and scaled to PLAN_SIZE
REF_WIDTH = 512
CLEARANCE_PX_REF = 12                # obstacle inflation radius (original CLEARANCE_PX)
START_OFFSET_REF = 10                # start = (h - 10, w // 2) in the original
GOAL_ROW_FRAC = 0.35                 # goal row as fraction of height (original 45/288 = 0.16 lies inside the
                                     # geometric map's blocked band at the top of real frames)
ENDPOINT_FALLBACK = True             # if the fixed start/goal row is blocked, search nearby rows (minimal addition)
SHARED_ENDPOINTS = False             # False: each method picks start/goal on its own map (as in the original planner)
GOAL_STICKINESS = 0.0                # >0 prefers a goal near the previous frame's goal (same rule for A, B, C).
                                     # 0 keeps Method A a pure single-frame baseline; try 1.0 as an ablation.

# Ego-vehicle hood at the bottom of the image: 'auto' (detected from the cost maps), a fraction of height, or 0 = off.
# Hood rows take the cost of the row just above, capped below the obstacle threshold: the vehicle itself is never an obstacle.
HOOD_FRAC = 'auto'
HOOD_ROWS = 0                        # full-resolution rows, set per scenario by set_hood()

# Display only (does not affect planning or metrics)
HORIZON_FRAC = 0.28                  # image row where the road vanishes, as a fraction of height
VEHICLE_WIDTH_FRAC = 0.50            # drawn corridor width at the bottom of the image (approx. vehicle width), fraction of width
FLOW_VIS_MAX = 4.0                   # flow (px) shown at full colour in the RAFT panel
PIPELINE_VIDEO = True                # also write pipeline.mp4 (semantic, geometric, cost maps, RAFT flow)

# Output
FPS = 15
RUN_TAG = ''                         # e.g. 'alpha04': results go to final_evaluation/alpha04/... so runs don't overwrite each other
N_VALIDATE = 5                       # frames shown by the validation cell
VALIDATE_FROM = 100                  # first frame index for validation (pick a part where the vehicle moves)

SCALE = PLAN_SIZE[0] / REF_WIDTH
CLEARANCE_PX = max(1, int(round(CLEARANCE_PX_REF * SCALE)))
START_OFFSET = max(1, int(round(START_OFFSET_REF * SCALE)))
GOAL_ROW = int(round(GOAL_ROW_FRAC * PLAN_SIZE[1]))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

assert PLAN_SIZE[0] % 8 == 0 and PLAN_SIZE[1] % 8 == 0, 'RAFT needs dimensions divisible by 8'
assert abs(PLAN_SIZE[0] / PLAN_SIZE[1] - 1024 / 544) < 1e-3, 'PLAN_SIZE should keep the 1024x544 aspect ratio'
print(f'Device: {DEVICE} | plan size {PLAN_SIZE} | clearance {CLEARANCE_PX}px | goal row {GOAL_ROW} | start offset {START_OFFSET}')
EVAL_DIR = ROOT / 'final_evaluation' / RUN_TAG if RUN_TAG else ROOT / 'final_evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
print('Results folder:', EVAL_DIR)
if DEVICE.type == 'cpu':
    print('Warning: no GPU. Select Runtime > Change runtime type > GPU for practical RAFT speed.')

## Data loading and validation

Frame IDs are parsed from filenames, so `frame_000131.jpg`, `outputs/frame_000131_semantic_cost.npy` and `outputs/frame_000131_geometric_cost.npy` are matched by the number 131, regardless of ZIP order or folder prefix.

In [ ]:
FRAME_RE = re.compile(r'^(?:frame_)?(\d+)(?:_(semantic|geometric)_cost)?\.(jpg|jpeg|png|npy)$', re.I)
KINDS = {'rgb': ('rgb.zip', ('jpg', 'jpeg', 'png')),
         'semantic': ('semantic_costs.zip', ('npy',)),
         'geometric': ('geometry_costs.zip', ('npy',))}


def index_zip(path, kind):
    """Map frame ID -> member name. Accepts names with or without the _semantic_cost/_geometric_cost suffix."""
    exts = KINDS[kind][1]
    index, skipped = {}, []
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            base = Path(name).name
            if name.endswith('/') or base.startswith('._') or '__MACOSX' in name:
                continue
            m = FRAME_RE.match(base)
            if not m or m.group(3).lower() not in exts:
                skipped.append(name)
                continue
            tag = (m.group(2) or '').lower()
            if (kind == 'rgb' and tag) or (kind != 'rgb' and tag and tag != kind):
                skipped.append(name)
                continue
            fid = int(m.group(1))
            if fid in index:
                raise ValueError(f'Duplicate frame {fid} in {path}: {index[fid]} and {name}')
            index[fid] = name
    return index, skipped


def local_copy(path):
    target = CACHE / path.parent.name / path.name
    if not target.exists() or target.stat().st_size != path.stat().st_size:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(path, target)
    return target


def get_data(folder, verbose=True, limit=True):
    paths = {k: folder / v[0] for k, v in KINDS.items()}
    missing = [str(p) for p in paths.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(f'Missing ZIPs: {missing}')
    indexes = {}
    for kind, p in paths.items():
        indexes[kind], skipped = index_zip(p, kind)
        if verbose:
            sample = indexes[kind][min(indexes[kind])] if indexes[kind] else '-'
            print(f'  {kind:9s} {len(indexes[kind]):4d} frames  e.g. {sample}' + (f'  ({len(skipped)} other files ignored, e.g. {skipped[0]})' if skipped else ''))
    sets = {k: set(v) for k, v in indexes.items()}
    union = set().union(*sets.values())
    problems = {k: sorted(union - s) for k, s in sets.items() if union - s}
    if not union:
        raise ValueError(f'No frame files recognised in {folder}')
    if problems:
        msg = '; '.join(f'{k} missing {len(v)} IDs (first: {v[:8]})' for k, v in problems.items())
        raise ValueError(f'Mismatched frame IDs in {folder}: {msg}')
    ids = sorted(union)
    gaps = [(a, b) for a, b in zip(ids[:-1], ids[1:]) if b != a + 1]
    if verbose:
        print(f'  IDs {ids[0]}..{ids[-1]}, {len(ids)} frames' + (f', {len(gaps)} gaps, first {gaps[:3]}' if gaps else ', consecutive'))
    if limit and MAX_FRAMES:
        ids = ids[:MAX_FRAMES]
    return paths, indexes, ids


def read_rgb(z, name):
    """Returns RGB at PLAN_SIZE and the original (h, w)."""
    bgr = cv2.imdecode(np.frombuffer(z.read(name), np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f'Could not decode {name}')
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return cv2.resize(rgb, PLAN_SIZE, interpolation=cv2.INTER_AREA), bgr.shape[:2]


def read_cost(z, name, rgb_hw):
    """Loads a full-resolution cost map and checks it is 2-D, finite, in [0, 1] and the same size as the RGB frame."""
    a = np.load(io.BytesIO(z.read(name)), allow_pickle=False)
    a = np.squeeze(a)
    if a.ndim != 2:
        raise ValueError(f'{name}: expected a 2-D map, got shape {a.shape}')
    a = a.astype(np.float32)
    if not np.isfinite(a).all():
        raise ValueError(f'{name}: contains NaN/inf')
    if a.min() < -1e-3 or a.max() > 1 + 1e-3:
        raise ValueError(f'{name}: values in [{a.min():.3f}, {a.max():.3f}], expected [0, 1] (0 = safe, 1 = unsafe)')
    if a.shape != tuple(rgb_hw):
        raise ValueError(f'{name}: shape {a.shape} does not match RGB {tuple(rgb_hw)}')
    return np.clip(a, 0, 1)


def combine(semantic, geometric):
    """Weighted average at full resolution. With HARD_OBSTACLES, any pixel that either model alone marks as an
    obstacle is set to 1.0, so averaging cannot turn an obstacle into free space. Downsampling keeps obstacles
    via an 'any' rule so thin obstacles are not lost."""
    if HOOD_ROWS:
        cap = 0.9 * OBSTACLE_THRESHOLD
        semantic, geometric = semantic.copy(), geometric.copy()
        semantic[-HOOD_ROWS:] = np.minimum(semantic[-HOOD_ROWS - 1], cap)
        geometric[-HOOD_ROWS:] = np.minimum(geometric[-HOOD_ROWS - 1], cap)
    cost = (SEMANTIC_WEIGHT * semantic + GEOMETRY_WEIGHT * geometric) / (SEMANTIC_WEIGHT + GEOMETRY_WEIGHT)
    small = cv2.resize(cost, PLAN_SIZE, interpolation=cv2.INTER_AREA)
    if HARD_OBSTACLES:
        hard = ((semantic >= OBSTACLE_THRESHOLD) | (geometric >= OBSTACLE_THRESHOLD)).astype(np.float32)
        hard_small = cv2.resize(hard, PLAN_SIZE, interpolation=cv2.INTER_AREA) > 0
        small = np.where(hard_small, 1.0, small)
    return small.astype(np.float32)


def detect_hood_rows(sem, geo):
    """Number of bottom rows where (almost) the whole row is an obstacle in either map."""
    full = (((sem >= OBSTACLE_THRESHOLD) | (geo >= OBSTACLE_THRESHOLD)).mean(axis=1) >= 0.9)
    k = 0
    while k < len(full) // 3 and full[-1 - k]:
        k += 1
    return k


def set_hood(zips, indexes, ids):
    global HOOD_ROWS
    _, hw = read_rgb(zips['rgb'], indexes['rgb'][ids[0]])
    if HOOD_FRAC == 'auto':
        rows = [detect_hood_rows(read_cost(zips['semantic'], indexes['semantic'][f], hw),
                                 read_cost(zips['geometric'], indexes['geometric'][f], hw)) for f in ids[:5]]
        k = int(np.median(rows))
        HOOD_ROWS = k + int(round(0.01 * hw[0])) if k else 0
    else:
        HOOD_ROWS = int(round(float(HOOD_FRAC) * hw[0]))
    print(f'  vehicle hood: bottom {HOOD_ROWS} of {hw[0]} rows excluded' + (' (auto)' if HOOD_FRAC == 'auto' else ''))


def load_frame(zips, indexes, fid):
    rgb, hw = read_rgb(zips['rgb'], indexes['rgb'][fid])
    sem = read_cost(zips['semantic'], indexes['semantic'][fid], hw)
    geo = read_cost(zips['geometric'], indexes['geometric'][fid], hw)
    return rgb, sem, geo, combine(sem, geo)

## A* Path Planning

The planner operates on image-space traversability cost maps, where higher costs indicate less traversable regions. Paths are represented as `(row, col)` coordinates.

The planning pipeline consists of:
- **`inflate_obstacles`**: Expands high-cost obstacle regions to provide additional safety clearance.
- **`select_start`, `select_goal`, `select_safe_goal`**: Select valid planning endpoints, with optional fallback positions when the predefined locations are blocked.
- **`astar_cost_map`**: Finds a low-cost path while avoiding inflated obstacles.
- **`smooth_path_safe`**: Reduces unnecessary turns while preserving collision constraints.
- **`draw_vehicle_corridor`**: Visualizes the planned path as a perspective driving corridor on the RGB frame.

Before planning, the vehicle hood region is excluded using `HOOD_FRAC`, while `GOAL_ROW_FRAC` controls the target goal row. The `ENDPOINT_FALLBACK` parameter enables alternative start and goal positions when necessary.

The `plan` function integrates these components into a single pipeline. The same planner and parameters are used for all three methods (independent A*, temporal fusion, and RAFT-aligned fusion), ensuring that differences in results come from temporal processing rather than changes to the planning algorithm.

In [ ]:
def astar_cost_map(cost_map, start, goal, obstacle_threshold=0.75):
    h, w = cost_map.shape

    if (
        cost_map[start] >= obstacle_threshold or
        cost_map[goal] >= obstacle_threshold
    ):
        return None

    neighbors = [
        (-1, 0), (1, 0),
        (0, -1), (0, 1),
        (-1, -1), (-1, 1),
        (1, -1), (1, 1)
    ]

    def heuristic(a, b):
        return math.hypot(a[0] - b[0], a[1] - b[1])

    heap = [(heuristic(start, goal), 0.0, start)]
    g_score = {start: 0.0}
    came_from = {}
    closed = set()

    while heap:
        _, current_g, current = heapq.heappop(heap)

        if current in closed:
            continue

        if current == goal:
            path = [current]
            while current in came_from:
                current = came_from[current]
                path.append(current)
            return np.asarray(path[::-1], dtype=np.int32)

        closed.add(current)
        r, c = current

        for dr, dc in neighbors:
            nr = r + dr
            nc = c + dc

            if not (0 <= nr < h and 0 <= nc < w):
                continue
            if cost_map[nr, nc] >= obstacle_threshold:
                continue

            # Prevent diagonal corner cutting.
            if dr != 0 and dc != 0:
                if (
                    cost_map[r + dr, c] >= obstacle_threshold or
                    cost_map[r, c + dc] >= obstacle_threshold
                ):
                    continue

            neighbor = (nr, nc)
            if neighbor in closed:
                continue

            step_distance = math.hypot(dr, dc)
            average_cost = (float(cost_map[r, c]) + float(cost_map[nr, nc])) / 2.0
            step_cost = step_distance * (1.0 + 5.0 * average_cost)
            tentative_g = current_g + step_cost

            if tentative_g < g_score.get(neighbor, float('inf')):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                priority = tentative_g + heuristic(neighbor, goal)
                heapq.heappush(heap, (priority, tentative_g, neighbor))

    return None


def inflate_obstacles(cost_map, clearance_px=12, threshold=0.75):
    obstacles = cost_map >= threshold
    radius = int(clearance_px)
    yy, xx = np.ogrid[-radius:radius + 1, -radius:radius + 1]
    disk = (xx * xx + yy * yy) <= radius * radius
    inflated = binary_dilation(obstacles, structure=disk)
    safe_cost = cost_map.copy()
    safe_cost[inflated] = 1.0
    return safe_cost, inflated


def select_safe_goal(cost_map, goal_row=45, threshold=0.75):
    h, w = cost_map.shape
    row = int(np.clip(goal_row, 0, h - 1))
    candidates = np.where(cost_map[row] < threshold)[0]
    if len(candidates) == 0:
        return None
    # Prefer low cost, but avoid unnecessary lateral deviation.
    scores = cost_map[row, candidates] + 0.15 * np.abs(candidates - w // 2) / w
    best_col = int(candidates[np.argmin(scores)])
    return (row, best_col)


def collision_free_segment(p1, p2, safe_cost, threshold=0.75):
    r1, c1 = map(int, p1)
    r2, c2 = map(int, p2)
    steps = max(abs(r2 - r1), abs(c2 - c1)) * 2 + 1
    rows = np.rint(np.linspace(r1, r2, steps)).astype(int)
    cols = np.rint(np.linspace(c1, c2, steps)).astype(int)
    h, w = safe_cost.shape
    if np.any((rows < 0) | (rows >= h) | (cols < 0) | (cols >= w)):
        return False
    return bool(np.all(safe_cost[rows, cols] < threshold))


def smooth_path_safe(path, safe_cost, threshold=0.75):
    if path is None or len(path) < 3:
        return path

    simplified = [path[0]]
    i = 0
    while i < len(path) - 1:
        next_i = i + 1
        for j in range(len(path) - 1, i, -1):
            if collision_free_segment(path[i], path[j], safe_cost, threshold):
                next_i = j
                break
        simplified.append(path[next_i])
        i = next_i

    # Dense samples along verified straight segments.
    dense = []
    for k in range(len(simplified) - 1):
        a = np.asarray(simplified[k], dtype=float)
        b = np.asarray(simplified[k + 1], dtype=float)
        n = max(2, int(np.ceil(np.linalg.norm(b - a) * 2)))
        segment = np.linspace(a, b, n)
        if k > 0:
            segment = segment[1:]
        dense.extend(segment)
    return np.asarray(dense, dtype=float)


def draw_vehicle_corridor(rgb, path, band_width=24, opacity=0.35):
    result = cv2.cvtColor(rgb.copy(), cv2.COLOR_RGB2BGR)
    if path is None or len(path) < 2:
        return cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    points = np.rint(path[:, ::-1]).astype(np.int32).reshape(-1, 1, 2)
    overlay = result.copy()
    cv2.polylines(overlay, [points], False, (0, 220, 0), band_width, cv2.LINE_AA)
    cv2.addWeighted(overlay, opacity, result, 1 - opacity, 0, dst=result)
    cv2.polylines(result, [points], False, (0, 255, 255), 2, cv2.LINE_AA)
    return cv2.cvtColor(result, cv2.COLOR_BGR2RGB)


def select_start(safe_cost, threshold=OBSTACLE_THRESHOLD):
    """Original start (h - START_OFFSET, w // 2); if blocked, the nearest free column on that row, then rows above."""
    h, w = safe_cost.shape
    row = h - START_OFFSET
    if safe_cost[row, w // 2] < threshold:
        return (row, w // 2)
    if not ENDPOINT_FALLBACK:
        return None
    for r in range(row, int(h * 0.75), -1):
        free = np.where(safe_cost[r] < threshold)[0]
        if len(free):
            return (r, int(free[np.argmin(np.abs(free - w // 2))]))
    return None


def goal_on_row(safe_cost, row, threshold, prev_goal):
    """Original select_safe_goal; with GOAL_STICKINESS > 0 an extra term prefers columns near the previous goal."""
    if prev_goal is None or GOAL_STICKINESS <= 0:
        return select_safe_goal(safe_cost, goal_row=row, threshold=threshold)
    h, w = safe_cost.shape
    row = int(np.clip(row, 0, h - 1))
    candidates = np.where(safe_cost[row] < threshold)[0]
    if len(candidates) == 0:
        return None
    scores = (safe_cost[row, candidates] + 0.15 * np.abs(candidates - w // 2) / w
              + GOAL_STICKINESS * np.abs(candidates - prev_goal[1]) / w)
    return (row, int(candidates[np.argmin(scores)]))


def select_goal(safe_cost, threshold=OBSTACLE_THRESHOLD, prev_goal=None):
    """Goal on GOAL_ROW; if that row is fully blocked, try rows closer to the vehicle."""
    goal = goal_on_row(safe_cost, GOAL_ROW, threshold, prev_goal)
    if goal is not None or not ENDPOINT_FALLBACK:
        return goal
    h = safe_cost.shape[0]
    for r in range(GOAL_ROW + 1, h // 2):
        goal = goal_on_row(safe_cost, r, threshold, prev_goal)
        if goal is not None:
            return goal
    return None


def plan(cost, start=None, goal=None, safe=None, prev_goal=None):
    """Identical planning for every method: inflate, pick endpoints if not given, A*, collision-checked smoothing.
    Returns (path, status, goal)."""
    if safe is None:
        safe, _ = inflate_obstacles(cost, CLEARANCE_PX, OBSTACLE_THRESHOLD)
    if start is None:
        start = select_start(safe)
    if goal is None:
        goal = select_goal(safe, prev_goal=prev_goal)
    if start is None or goal is None:
        return None, 'no_endpoint', goal
    if safe[start] >= OBSTACLE_THRESHOLD or safe[goal] >= OBSTACLE_THRESHOLD:
        return None, 'endpoint_blocked', goal
    raw = astar_cost_map(safe, start, goal, OBSTACLE_THRESHOLD)
    if raw is None:
        return None, 'no_path', goal
    return smooth_path_safe(raw, safe, OBSTACLE_THRESHOLD), 'ok', goal

## RAFT, warping and temporal fusion

`raft(current, previous)` gives backward flow: for each current-frame pixel `x`, its position in the previous frame is `x + flow(x)`. The previous cost map is sampled there with `cv2.remap`.
Pixels are rejected when they sample outside the image or, with `FB_CHECK`, when forward and backward flow disagree (occlusions, unreliable matches). Rejected pixels keep the current cost.

In [ ]:
raft_weights = Raft_Small_Weights.DEFAULT
raft_model = raft_small(weights=raft_weights, progress=True).to(DEVICE).eval()
raft_transforms = raft_weights.transforms()
GRID_X, GRID_Y = np.meshgrid(np.arange(PLAN_SIZE[0], dtype=np.float32), np.arange(PLAN_SIZE[1], dtype=np.float32))


def to_tensor(rgb):
    return torch.from_numpy(np.ascontiguousarray(rgb)).permute(2, 0, 1).float().div(255).unsqueeze(0)


@torch.inference_mode()
def compute_flows(current_rgb, previous_rgb):
    """Backward flow (current -> previous) and, if FB_CHECK, forward flow (previous -> current), in one batch."""
    cur, prev = to_tensor(current_rgb), to_tensor(previous_rgb)
    img1, img2 = (torch.cat([cur, prev]), torch.cat([prev, cur])) if FB_CHECK else (cur, prev)
    img1, img2 = raft_transforms(img1, img2)
    flows = raft_model(img1.to(DEVICE), img2.to(DEVICE), num_flow_updates=RAFT_ITERS)[-1]
    flows = flows.permute(0, 2, 3, 1).float().cpu().numpy()
    return flows[0], (flows[1] if FB_CHECK else None)


def warp_previous(previous, backward, forward=None):
    """Sample `previous` at current-frame coordinates. Returns warped map, validity mask and FB-consistent fraction."""
    h, w = previous.shape
    mx = GRID_X + backward[..., 0]
    my = GRID_Y + backward[..., 1]
    valid = (mx >= 0) & (mx <= w - 1) & (my >= 0) & (my <= h - 1)
    warped = cv2.remap(previous.astype(np.float32), mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    fb_ok = np.ones_like(valid)
    if forward is not None:
        fwd_at = np.stack([cv2.remap(forward[..., i], mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
                           for i in range(2)], -1)
        err = np.sum((backward + fwd_at) ** 2, -1)
        fb_ok = err < 0.01 * (np.sum(backward ** 2, -1) + np.sum(fwd_at ** 2, -1)) + 0.5
        valid &= fb_ok
    return warped, valid, float(fb_ok.mean())


def fuse(current, previous, valid=None):
    if previous is None:
        return current.copy()
    fused = FUSION_ALPHA * current + (1 - FUSION_ALPHA) * previous
    if valid is not None:
        fused = np.where(valid, fused, current)
    if PRESERVE_CURRENT_OBSTACLES:
        fused = np.where(current >= OBSTACLE_THRESHOLD, current, fused)
    return np.clip(fused, 0, 1).astype(np.float32)


print('RAFT loaded on', DEVICE)

## Metrics and visualisation

Cost, clearance and obstacle-crossing metrics for **all three methods** are computed on the same reference: the current frame's combined map (Method A's input). Clearance is the distance to raw obstacles (cost ≥ threshold), not to the inflated ones.

In [ ]:
METHODS = ('A', 'B', 'C')
LABELS = {'A': 'A: current map + A*', 'B': 'B: temporal fusion + A*', 'C': 'C: RAFT fusion + A*'}


def sample_path(path, spacing=0.5):
    d = np.r_[0, np.cumsum(np.linalg.norm(np.diff(path, axis=0), axis=1))]
    if d[-1] < 1e-8:
        return path[:1]
    s = np.r_[np.arange(0, d[-1], spacing), d[-1]]
    return np.column_stack([np.interp(s, d, path[:, 0]), np.interp(s, d, path[:, 1])])


def path_x_by_row(path, h):
    """Horizontal position of the path at each image row (from the original notebook)."""
    rows = np.rint(path[:, 0]).astype(int)
    cols = path[:, 1]
    unique_rows = np.unique(rows)
    x_values = np.array([cols[rows == r].mean() for r in unique_rows])
    result = np.full(h, np.nan)
    lo, hi = int(unique_rows.min()), int(unique_rows.max())
    result[lo:hi + 1] = np.interp(np.arange(lo, hi + 1), unique_rows, x_values)
    return result


def temporal_deviation(previous, current, h):
    if previous is None or current is None:
        return np.nan
    a, b = path_x_by_row(previous, h), path_x_by_row(current, h)
    ok = np.isfinite(a) & np.isfinite(b)
    return float(np.mean(np.abs(a[ok] - b[ok]))) if ok.sum() >= 3 else np.nan


def reference(current):
    obstacles = current >= OBSTACLE_THRESHOLD
    _, inflated = inflate_obstacles(current, CLEARANCE_PX, OBSTACLE_THRESHOLD)
    return dict(cost=current, obstacles=obstacles, inflated=inflated,
                clearance=distance_transform_edt(~obstacles).astype(np.float32))


def path_metrics(path, ref, previous_path):
    nan = dict(length_px=np.nan, mean_ref_cost=np.nan, max_ref_cost=np.nan, min_clearance_px=np.nan,
               mean_clearance_px=np.nan, crosses_obstacle=np.nan, violates_margin=np.nan, turning_deg=np.nan,
               temporal_deviation_px=np.nan, goal_shift_px=np.nan)
    if path is None or len(path) < 2:
        return nan
    h, w = ref['cost'].shape
    s = sample_path(path)
    r = np.clip(np.rint(s[:, 0]).astype(int), 0, h - 1)
    c = np.clip(np.rint(s[:, 1]).astype(int), 0, w - 1)
    coarse = sample_path(path, spacing=2.0)
    steps = np.diff(coarse[:, ::-1], axis=0)
    turning = np.degrees(np.abs(np.diff(np.unwrap(np.arctan2(steps[:, 1], steps[:, 0]))))).sum() if len(steps) > 1 else 0.0
    return dict(length_px=float(np.linalg.norm(np.diff(path, axis=0), axis=1).sum()),
                mean_ref_cost=float(ref['cost'][r, c].mean()), max_ref_cost=float(ref['cost'][r, c].max()),
                min_clearance_px=float(ref['clearance'][r, c].min()), mean_clearance_px=float(ref['clearance'][r, c].mean()),
                crosses_obstacle=float(ref['obstacles'][r, c].any()),
                violates_margin=float(ref['clearance'][r, c].min() < CLEARANCE_PX - 1.0),   # > 1 px inside the margin
                turning_deg=float(turning), temporal_deviation_px=temporal_deviation(previous_path, path, h),
                goal_shift_px=float(np.linalg.norm(path[-1] - previous_path[-1])) if previous_path is not None else np.nan)


METHOD_COLORS = {'A': (180, 119, 31), 'B': (14, 127, 255), 'C': (44, 160, 44)}   # BGR, same colours as the plots
COST_LUT = (plt.get_cmap('RdYlGn_r')(np.linspace(0, 1, 256))[:, 2::-1] * 255).astype(np.uint8)
FONT = cv2.FONT_HERSHEY_DUPLEX


def corridor_polygon(path, h, w):
    """Centreline and edges of a corridor of constant real width on flat ground: its image width shrinks
    linearly towards the horizon. Corners are rounded slightly for display only."""
    xy = path[:, ::-1].astype(np.float64)
    keep = np.r_[True, np.linalg.norm(np.diff(xy, axis=0), axis=1) > 0.5]
    xy = xy[keep]
    if len(xy) < 2:
        return None
    d = np.r_[0, np.cumsum(np.linalg.norm(np.diff(xy, axis=0), axis=1))]
    s = np.linspace(0, d[-1], max(2, int(d[-1] / 2)))
    xy = np.column_stack([np.interp(s, d, xy[:, 0]), np.interp(s, d, xy[:, 1])])
    if len(xy) > 5:
        xy[1:-1] = gaussian_filter1d(xy, 3, axis=0, mode='nearest')[1:-1]
    t = np.gradient(xy, axis=0)
    t /= np.maximum(np.linalg.norm(t, axis=1, keepdims=True), 1e-6)
    n = np.column_stack([-t[:, 1], t[:, 0]])
    horizon = HORIZON_FRAC * h
    half = 0.5 * VEHICLE_WIDTH_FRAC * w * np.clip((xy[:, 1] - horizon) / (h - 1 - horizon), 0.02, 1.0)
    return xy, xy + n * half[:, None], xy - n * half[:, None]


def pts(a):
    return np.rint(a).astype(np.int32).reshape(-1, 1, 2)


def draw_corridor(img, path, fill=(70, 200, 60), centre=(0, 230, 255), alpha=0.38):
    """Perspective driving corridor on a BGR image: translucent fill, edges, centreline and goal marker."""
    if path is None or len(path) < 2:
        return img
    poly = corridor_polygon(path, *img.shape[:2])
    if poly is None:
        return img
    c, left, right = poly
    overlay = img.copy()
    cv2.fillPoly(overlay, [pts(np.vstack([left, right[::-1]]))], fill, cv2.LINE_AA)
    img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
    edge = tuple(min(255, int(v * 1.2) + 50) for v in fill)
    cv2.polylines(img, [pts(left), pts(right)], False, edge, 2, cv2.LINE_AA)
    cv2.polylines(img, [pts(c)], False, centre, 2, cv2.LINE_AA)
    g = tuple(int(v) for v in np.rint(c[-1]))
    cv2.circle(img, g, 5, centre, -1, cv2.LINE_AA)
    cv2.circle(img, g, 5, (30, 30, 30), 1, cv2.LINE_AA)
    return img


def label(img, text):
    img[:26] = (img[:26] * 0.3 + 18 * 0.7).astype(np.uint8)
    cv2.putText(img, text, (10, 18), FONT, 0.5, (245, 245, 245), 1, cv2.LINE_AA)
    return img


def no_path(img, status):
    if status != 'ok':
        text = f'NO PATH  ({status})'
        (tw, th), _ = cv2.getTextSize(text, FONT, 0.5, 1)
        y = img.shape[0] - 12
        cv2.rectangle(img, (8, y - th - 8), (18 + tw, y + 6), (40, 40, 200), -1)
        cv2.putText(img, text, (13, y), FONT, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return img


def legend(img, items, corner='bl'):
    """items: list of (text, BGR colour) drawn as short lines in a dark box."""
    h, w = img.shape[:2]
    bw, bh = 150, 10 + 18 * len(items)
    x0, y0 = (8, h - bh - 8) if corner == 'bl' else (w - bw - 8, h - bh - 8)
    img[y0:y0 + bh, x0:x0 + bw] = (img[y0:y0 + bh, x0:x0 + bw] * 0.35 + 18 * 0.65).astype(np.uint8)
    for k, (text, colour) in enumerate(items):
        y = y0 + 14 + 18 * k
        cv2.line(img, (x0 + 8, y), (x0 + 30, y), colour, 3, cv2.LINE_AA)
        cv2.putText(img, text, (x0 + 38, y + 4), FONT, 0.4, (235, 235, 235), 1, cv2.LINE_AA)
    return img


def cost_panel(bgr, cost, title, alpha=0.6):
    coloured = COST_LUT[np.clip(cost * 255, 0, 255).astype(np.uint8)]
    img = cv2.addWeighted(coloured, alpha, bgr, 1 - alpha, 0)
    h, w = img.shape[:2]
    x0, y0, bw = w - 128, h - 22, 110                      # colour bar: 0 = safe, 1 = unsafe
    img[y0 - 16:y0 + 14, x0 - 8:x0 + bw + 8] = (img[y0 - 16:y0 + 14, x0 - 8:x0 + bw + 8] * 0.35 + 18 * 0.65).astype(np.uint8)
    img[y0:y0 + 8, x0:x0 + bw] = COST_LUT[np.linspace(0, 255, bw).astype(np.uint8)][None]
    cv2.putText(img, 'safe', (x0, y0 - 4), FONT, 0.35, (230, 230, 230), 1, cv2.LINE_AA)
    cv2.putText(img, 'unsafe', (x0 + bw - 40, y0 - 4), FONT, 0.35, (230, 230, 230), 1, cv2.LINE_AA)
    return label(img, title)


def flow_panel(flow, h, w):
    """Standard optical-flow colouring: hue = direction, saturation = speed, white = no motion."""
    if flow is None:
        img = np.full((h, w, 3), 245, np.uint8)
        cv2.putText(img, 'no previous frame', (w // 2 - 80, h // 2), FONT, 0.5, (120, 120, 120), 1, cv2.LINE_AA)
        return label(img, 'RAFT optical flow')
    mag, ang = cv2.cartToPolar(flow[..., 0].astype(np.float32), flow[..., 1].astype(np.float32), angleInDegrees=True)
    hsv = np.empty((h, w, 3), np.uint8)
    hsv[..., 0] = (ang / 2).astype(np.uint8)
    hsv[..., 1] = np.clip(mag / FLOW_VIS_MAX * 255, 0, 255).astype(np.uint8)
    hsv[..., 2] = 255
    img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    r = 18                                                  # colour wheel legend
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    wm, wa = cv2.cartToPolar(xx.astype(np.float32), yy.astype(np.float32), angleInDegrees=True)
    wheel = cv2.cvtColor(np.dstack([(wa / 2).astype(np.uint8), np.clip(wm / r * 255, 0, 255).astype(np.uint8),
                                    np.full_like(wm, 255, np.uint8)]), cv2.COLOR_HSV2BGR)
    inside = wm <= r
    cy, cx = h - r - 10, w - r - 10
    img[cy - r:cy + r + 1, cx - r:cx + r + 1][inside] = wheel[inside]
    cv2.circle(img, (cx, cy), r, (90, 90, 90), 1, cv2.LINE_AA)
    cv2.putText(img, f'mean {mag.mean():.2f}px', (10, h - 12), FONT, 0.45, (60, 60, 60), 1, cv2.LINE_AA)
    return label(img, 'RAFT optical flow (current -> previous)')


def all_paths(bgr, paths):
    img = bgr.copy()
    for m in METHODS:
        if paths[m] is not None and len(paths[m]) > 1:
            cv2.polylines(img, [pts(paths[m][:, ::-1])], False, (20, 20, 20), 5, cv2.LINE_AA)
            cv2.polylines(img, [pts(paths[m][:, ::-1])], False, METHOD_COLORS[m], 3, cv2.LINE_AA)
    return legend(img, [(f'{m}: {LABELS[m][3:].split(" +")[0]}', METHOD_COLORS[m]) for m in METHODS])


def grid(panels, cols, footer=None):
    rows = [np.hstack(panels[i:i + cols]) for i in range(0, len(panels), cols)]
    img = np.vstack(rows)
    h, w = panels[0].shape[:2]
    for k in range(1, cols):
        img[:, k * w - 1:k * w + 1] = 18
    for k in range(1, len(rows)):
        img[k * h - 1:k * h + 1] = 18
    if footer:
        bar = np.full((28, img.shape[1], 3), 18, np.uint8)
        cv2.putText(bar, footer, (10, 19), FONT, 0.5, (220, 220, 220), 1, cv2.LINE_AA)
        img = np.vstack([img, bar])
    return img


def comparison_frame(rgb, paths, status, footer):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    panels = [label(all_paths(bgr, paths), 'RGB + all paths')]
    for m in METHODS:
        panels.append(no_path(label(draw_corridor(bgr.copy(), paths[m], fill=METHOD_COLORS[m], centre=(255, 255, 255)),
                                    LABELS[m]), status[m]))
    return grid(panels, 2, footer)


def pipeline_frame(rgb, sem, geo, current, fused_c, flow, paths, status, footer):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    h, w = bgr.shape[:2]
    small = lambda a: cv2.resize(a.astype(np.float32), PLAN_SIZE, interpolation=cv2.INTER_AREA)
    planned = all_paths(draw_corridor(bgr.copy(), paths['C'], fill=METHOD_COLORS['C'], centre=(255, 255, 255)), paths)
    panels = [cost_panel(bgr, small(sem), 'Semantic cost'),
              cost_panel(bgr, small(geo), 'Geometric cost'),
              cost_panel(bgr, current, 'Combined cost (current frame)'),
              flow_panel(flow, h, w),
              cost_panel(bgr, fused_c, 'RAFT-fused cost (method C)'),
              no_path(label(planned, 'Planned corridor (C) + all paths'), status['C'])]
    return grid(panels, 3, footer)


def summarize(df):
    """Per-method summary. Temporal deviation is also reported on frames where all three methods have a value."""
    df = df.copy()
    key = ['scenario', 'frame_id']
    wide = df.pivot_table(index=key, columns='method', values='temporal_deviation_px')
    common = wide.dropna().index
    df['_common'] = df.set_index(key).index.isin(common)
    rows = []
    for m, part in df.groupby('method'):
        ok = part[part.success == 1]
        rows.append(dict(method=m, frames=len(part), success_rate=part.success.mean(),
                         mean_length_px=ok.length_px.mean(), mean_ref_cost=ok.mean_ref_cost.mean(),
                         mean_min_clearance_px=ok.min_clearance_px.mean(), mean_clearance_px=ok.mean_clearance_px.mean(),
                         obstacle_crossing_rate=ok.crosses_obstacle.mean(), margin_violation_rate=ok.violates_margin.mean(),
                         mean_turning_deg=ok.turning_deg.mean(), mean_goal_shift_px=part.goal_shift_px.mean(),
                         temporal_dev_common_px=part[part._common].temporal_deviation_px.mean(),
                         temporal_dev_all_px=part.temporal_deviation_px.mean(), n_common_transitions=int(part._common.sum()),
                         mean_raft_ms=part.raft_ms.mean(), mean_fusion_ms=part.fusion_ms.mean(), mean_plan_ms=part.plan_ms.mean()))
    return pd.DataFrame(rows)

## Validation on a few real frames - run this before the full evaluation

Checks: ZIP matching, cost-map shapes/ranges, the 0 = safe / 1 = unsafe convention, visual alignment with RGB, RAFT direction (warping the previous RGB with the flow must reduce the photometric error), and that B and C produce different maps.

In [ ]:
def validate(scenario=SELECTED_SCENARIO, n=N_VALIDATE):
    folder = ROOT / 'datasets' / scenario
    print('Scenario:', folder)
    paths, indexes, ids = get_data(folder, limit=False)
    ids = ids[VALIDATE_FROM:VALIDATE_FROM + n] or ids[:n]
    print(f'  validating frames {ids[0]}..{ids[-1]}')
    zips = {k: zipfile.ZipFile(local_copy(p)) for k, p in paths.items()}
    set_hood(zips, indexes, ids)
    stats, frames = [], []
    try:
        for fid in ids:
            rgb, hw = read_rgb(zips['rgb'], indexes['rgb'][fid])
            row = dict(frame_id=fid, rgb_hw=hw)
            maps = {}
            for kind in ('semantic', 'geometric'):
                a = np.squeeze(np.load(io.BytesIO(zips[kind].read(indexes[kind][fid])), allow_pickle=False))
                row[f'{kind}_shape'], row[f'{kind}_dtype'] = a.shape, str(a.dtype)
                row[f'{kind}_min'], row[f'{kind}_max'], row[f'{kind}_mean'] = float(np.nanmin(a)), float(np.nanmax(a)), float(np.nanmean(a))
                row[f'{kind}_obst_frac'] = float((a >= OBSTACLE_THRESHOLD).mean())
                h = a.shape[0]
                row[f'{kind}_bottom_mean'] = float(np.nanmean(a[int(.8 * h):, a.shape[1] // 3: 2 * a.shape[1] // 3]))
                row[f'{kind}_top_mean'] = float(np.nanmean(a[:int(.15 * h)]))
                maps[kind] = a
            stats.append(row)
            frames.append((fid, rgb, hw, maps))
    finally:
        for z in zips.values():
            z.close()

    table = pd.DataFrame(stats)
    with pd.option_context('display.max_columns', None, 'display.width', 250):
        display(table)
    for kind in ('semantic', 'geometric'):
        if (table[f'{kind}_bottom_mean'] > table[f'{kind}_top_mean']).mean() > 0.5:
            print(f'WARNING: {kind} cost is higher in front of the vehicle than at the top of the image. '
                  'Check the convention is 0 = safe, 1 = unsafe.')

    # Strict checks used by the pipeline
    for fid, rgb, hw, maps in frames:
        for kind in ('semantic', 'geometric'):
            a = maps[kind].astype(np.float32)
            if a.shape != tuple(hw) or not np.isfinite(a).all() or a.min() < -1e-3 or a.max() > 1 + 1e-3:
                print(f'FAIL frame {fid} {kind}: shape {a.shape} vs {hw}, range [{np.nanmin(a):.3f}, {np.nanmax(a):.3f}]')

    # Visual alignment and planning
    fig, axes = plt.subplots(len(frames), 4, figsize=(20, 3.2 * len(frames)), squeeze=False)
    for i, (fid, rgb, hw, maps) in enumerate(frames):
        sem = np.clip(maps['semantic'].astype(np.float32), 0, 1)
        geo = np.clip(maps['geometric'].astype(np.float32), 0, 1)
        current = combine(sem, geo)
        path, status, _ = plan(current)
        for ax, (img, title) in zip(axes[i], [(cv2.resize(sem, PLAN_SIZE), 'semantic'), (cv2.resize(geo, PLAN_SIZE), 'geometric'),
                                              (current, 'combined'), (None, f'A* ({status})')]):
            ax.imshow(rgb)
            if img is not None:
                ax.imshow(img, cmap='RdYlGn_r', vmin=0, vmax=1, alpha=0.5)
            else:
                ax.imshow(draw_vehicle_corridor(rgb, path, 2 * CLEARANCE_PX))
            ax.set_title(f'frame {fid}: {title}')
            ax.axis('off')
    plt.tight_layout()
    plt.show()

    # RAFT direction check and B vs C difference
    if len(frames) >= 2:
        print('\nRAFT check (photometric error should drop after warping):')
        prev_rgb, prev_cost = frames[0][1], combine(*(np.clip(frames[0][3][k].astype(np.float32), 0, 1) for k in ('semantic', 'geometric')))
        for fid, rgb, hw, maps in frames[1:]:
            cur_cost = combine(*(np.clip(maps[k].astype(np.float32), 0, 1) for k in ('semantic', 'geometric')))
            bwd, fwd = compute_flows(rgb, prev_rgb)
            gray_prev = cv2.cvtColor(prev_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
            gray_cur = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
            warped_img, valid, fb = warp_previous(gray_prev, bwd, fwd)
            if not valid.any():
                print(f'  frame {fid}: no valid warped pixels (flow {np.linalg.norm(bwd, axis=-1).mean():.2f}px) - check RAFT weights/inputs')
                prev_rgb, prev_cost = rgb, cur_cost
                continue
            e0 = np.abs(gray_cur - gray_prev)[valid].mean()
            e1 = np.abs(gray_cur - warped_img)[valid].mean()
            warped_cost, valid_c, _ = warp_previous(prev_cost, bwd, fwd)
            b, c = fuse(cur_cost, prev_cost), fuse(cur_cost, warped_cost, valid_c)
            flow_px = np.linalg.norm(bwd, axis=-1).mean()
            verdict = 'too little motion to test' if flow_px < 0.5 else ('OK' if e1 < e0 else 'CHECK FLOW DIRECTION')
            print(f'  frame {fid}: flow {flow_px:.2f}px | error unwarped {e0:.2f} -> warped {e1:.2f} '
                  f'{verdict} | valid {valid.mean():.0%} (FB {fb:.0%}) | '
                  f'mean|B-C| {np.abs(b - c).mean():.4f} | mean|A-B| {np.abs(cur_cost - b).mean():.4f}')
            prev_rgb, prev_cost = rgb, cur_cost


validate()

## Full evaluation

In [ ]:
def json_safe(v):
    return list(v) if isinstance(v, tuple) else v


def run(folder):
    print('\n' + folder.name)
    paths, indexes, ids = get_data(folder)
    out = EVAL_DIR / folder.name
    out.mkdir(parents=True, exist_ok=True)
    keys = ['RUN_MODE', 'MAX_FRAMES', 'PLAN_SIZE', 'SEMANTIC_WEIGHT', 'GEOMETRY_WEIGHT', 'OBSTACLE_THRESHOLD', 'HARD_OBSTACLES',
            'FUSION_ALPHA', 'FUSION_HISTORY', 'PRESERVE_CURRENT_OBSTACLES', 'RESET_ON_GAP', 'RAFT_ITERS', 'FB_CHECK',
            'CLEARANCE_PX_REF', 'START_OFFSET_REF', 'GOAL_ROW_FRAC', 'ENDPOINT_FALLBACK', 'SHARED_ENDPOINTS',
            'CLEARANCE_PX', 'START_OFFSET', 'GOAL_ROW', 'HOOD_FRAC', 'GOAL_STICKINESS', 'RUN_TAG', 'FPS']
    cfg = {k: json_safe(globals()[k]) for k in keys}
    cfg.update(scenario=folder.name, frame_ids=[int(ids[0]), int(ids[-1])], n_frames=len(ids), device=str(DEVICE),
               torch=torch.__version__, torchvision=torchvision.__version__, raft='raft_small / Raft_Small_Weights.DEFAULT')
    (out / 'config.json').write_text(json.dumps(cfg, indent=2))

    zips = {k: zipfile.ZipFile(local_copy(p)) for k, p in paths.items()}
    set_hood(zips, indexes, ids)
    cfg['HOOD_ROWS'] = HOOD_ROWS
    (out / 'config.json').write_text(json.dumps(cfg, indent=2))
    videos = {}
    rows, diag = [], []
    saved = {m: np.empty(len(ids), dtype=object) for m in METHODS}
    endpoints = np.full((len(ids), 4), -1, np.int32)
    prev = dict(fid=None, rgb=None, B=None, C=None, paths={m: None for m in METHODS}, goals={m: None for m in METHODS})
    try:
        for i, fid in enumerate(tqdm(ids, desc=folder.name)):
            rgb, sem, geo, current = load_frame(zips, indexes, fid)
            if RESET_ON_GAP and prev['fid'] is not None and fid != prev['fid'] + 1:
                print(f'  gap {prev["fid"]} -> {fid}: temporal history reset')
                prev.update(rgb=None, B=None, C=None, paths={m: None for m in METHODS}, goals={m: None for m in METHODS})

            # B: same fusion, no alignment
            t = time.perf_counter()
            fused_b = fuse(current, prev['B'])
            fusion_b = time.perf_counter() - t

            # C: same fusion after RAFT alignment
            raft_ms, valid_frac, fb_frac, flow_mag = np.nan, np.nan, np.nan, np.nan
            if prev['rgb'] is None:
                fused_c, fusion_c = current.copy(), 0.0
            else:
                t = time.perf_counter()
                bwd, fwd = compute_flows(rgb, prev['rgb'])
                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()
                raft_ms = 1000 * (time.perf_counter() - t)
                t = time.perf_counter()
                warped, valid, fb_frac = warp_previous(prev['C'], bwd, fwd)
                fused_c = fuse(current, warped, valid)
                fusion_c = time.perf_counter() - t
                valid_frac, flow_mag = float(valid.mean()), float(np.linalg.norm(bwd, axis=-1).mean())

            ref = reference(current)
            safe_a, _ = inflate_obstacles(current, CLEARANCE_PX, OBSTACLE_THRESHOLD)
            start = goal = None
            if SHARED_ENDPOINTS:
                start, goal = select_start(safe_a), select_goal(safe_a)
                if start is not None:
                    endpoints[i, :2] = start
                if goal is not None:
                    endpoints[i, 2:] = goal

            maps = {'A': current, 'B': fused_b, 'C': fused_c}
            fusion_ms = {'A': 0.0, 'B': 1000 * fusion_b, 'C': 1000 * fusion_c}
            paths_now, status_now = {}, {}
            for m in METHODS:
                t = time.perf_counter()
                path, status, used_goal = plan(maps[m], start, goal, prev_goal=prev['goals'][m])
                prev['goals'][m] = used_goal if path is not None else prev['goals'][m]
                plan_ms = 1000 * (time.perf_counter() - t)
                rows.append(dict(scenario=folder.name, frame_id=fid, method=m, success=int(path is not None), status=status,
                                 **path_metrics(path, ref, prev['paths'][m]),
                                 raft_ms=raft_ms if m == 'C' else 0.0, fusion_ms=fusion_ms[m], plan_ms=plan_ms))
                saved[m][i] = path.astype(np.float32) if path is not None else np.empty((0, 2), np.float32)
                prev['paths'][m] = path
                paths_now[m], status_now[m] = path, status

            diag.append(dict(frame_id=fid, mean_abs_A_B=float(np.abs(current - fused_b).mean()),
                             mean_abs_A_C=float(np.abs(current - fused_c).mean()), mean_abs_B_C=float(np.abs(fused_b - fused_c).mean()),
                             warp_valid_frac=valid_frac, fb_consistent_frac=fb_frac, mean_flow_px=flow_mag,
                             obstacle_frac=float(ref['obstacles'].mean())))

            motion = f'flow {flow_mag:.2f}px  |  warp valid {valid_frac:.0%}' if prev['rgb'] is not None else 'first frame'
            footer = f'{folder.name}   frame {fid}   |   {motion}   |   ' + '   '.join(f'{m}: {status_now[m]}' for m in METHODS)
            frames = {'comparison': comparison_frame(rgb, paths_now, status_now, footer)}
            if PIPELINE_VIDEO:
                frames['pipeline'] = pipeline_frame(rgb, sem, geo, current, fused_c, bwd if prev['rgb'] is not None else None,
                                                    paths_now, status_now, footer)
            for name, img in frames.items():
                if name not in videos:
                    videos[name] = cv2.VideoWriter(str(CACHE / f'{folder.name}_{name}.mp4'), cv2.VideoWriter_fourcc(*'mp4v'),
                                                   FPS, (img.shape[1], img.shape[0]))
                videos[name].write(img)

            history_b = fused_b if FUSION_HISTORY == 'fused' else current
            history_c = fused_c if FUSION_HISTORY == 'fused' else current
            prev.update(fid=fid, rgb=rgb, B=history_b, C=history_c)
    finally:
        for v in videos.values():
            v.release()
        for z in zips.values():
            z.close()

    # Browser-playable H.264 copy in Drive
    for name in videos:
        raw, final = CACHE / f'{folder.name}_{name}.mp4', out / f'{name}.mp4'
        result = subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw), '-vcodec', 'libx264',
                                 '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(final)])
        if result.returncode != 0:
            shutil.copy(raw, final)

    df = pd.DataFrame(rows).merge(pd.DataFrame(diag), on='frame_id', how='left')
    df.to_csv(out / 'frame_metrics.csv', index=False)
    summary = summarize(df)
    summary.insert(0, 'scenario', folder.name)
    d = pd.DataFrame(diag)
    summary['mean_abs_B_C'] = d.mean_abs_B_C.mean()
    summary['mean_warp_valid_frac'] = d.warp_valid_frac.mean()
    summary.to_csv(out / 'summary.csv', index=False)
    np.savez_compressed(out / 'paths.npz', frame_ids=np.asarray(ids), endpoints_rc=endpoints,
                        **{f'paths_{m}': saved[m] for m in METHODS})

    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    plots = [('success', 'Success (rolling 10)'), ('temporal_deviation_px', 'Frame-to-frame deviation (px, rolling 10)'),
             ('min_clearance_px', 'Min clearance to current obstacles (px)'), ('mean_ref_cost', 'Mean cost on current map'),
             ('turning_deg', 'Total turning (deg)')]
    for ax, (col, title) in zip(axes.flat, plots):
        for m, part in df.groupby('method'):
            ax.plot(part.frame_id, part[col].rolling(10, min_periods=1).mean(), label=LABELS[m])
        ax.set_title(title); ax.set_xlabel('Frame'); ax.grid(alpha=0.25)
    axes.flat[0].legend(fontsize=8)
    ax = axes.flat[5]
    x = np.arange(3)
    bottom = np.zeros(3)
    for col in ('raft_ms', 'fusion_ms', 'plan_ms'):
        vals = summary.set_index('method').loc[list(METHODS), f'mean_{col}'].fillna(0).values
        ax.bar(x, vals, bottom=bottom, label=col.replace('_ms', ''))
        bottom += vals
    ax.set_xticks(x, METHODS); ax.set_title('Mean runtime per frame (ms)'); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(out / 'metrics_plot.png', dpi=150); plt.close(fig)

    print('Saved to', out)
    print(f'mean |B-C| = {d.mean_abs_B_C.mean():.4f}, warp valid = {d.warp_valid_frac.mean():.0%}'
          + ('  <- B and C are nearly identical' if d.mean_abs_B_C.mean() < 1e-3 else ''))
    display(summary.round(3))
    return df


datasets = ROOT / 'datasets'
scenarios = [datasets / SELECTED_SCENARIO] if RUN_MODE == 'single' else sorted(p for p in datasets.glob('scenario_*') if p.is_dir())
if not scenarios:
    raise FileNotFoundError(f'No scenario folders in {datasets}')

if RUN_MAIN_EXPERIMENT:
    print('Checking all scenarios before processing:')
    for folder in scenarios:
        print(folder.name)
        get_data(folder)

    all_frames = pd.concat([run(folder) for folder in scenarios], ignore_index=True)
    per_scenario = pd.concat([summarize(part).assign(scenario=name) for name, part in all_frames.groupby('scenario')], ignore_index=True)
    pooled = summarize(all_frames).assign(scenario='ALL')
    all_summary = pd.concat([per_scenario, pooled], ignore_index=True)
    all_summary = all_summary[['scenario'] + [c for c in all_summary.columns if c != 'scenario']]
    all_summary.to_csv(EVAL_DIR / 'all_scenarios_summary.csv', index=False)
    display(all_summary.round(3))

## Statistics across scenarios

Paired tests on the same frames: Wilcoxon signed-rank for continuous metrics, exact McNemar for yes/no metrics (margin violation, obstacle crossing). Reported for all frames and for frames where the vehicle moves (`mean_flow_px > 1`), per scenario and pooled (`ALL`). Saves `statistics.csv` and `comparison_by_scenario.png`. This cell can be run on its own later: it reloads the saved `frame_metrics.csv` files.

In [ ]:
from scipy.stats import wilcoxon, binomtest

MOVING_FLOW_PX = 1.0                 # "moving" frames: mean RAFT flow above this
CONTINUOUS = ['temporal_deviation_px', 'goal_shift_px', 'turning_deg', 'min_clearance_px']
BINARY = ['violates_margin', 'crosses_obstacle']
PAIRS = [('A', 'B'), ('A', 'C'), ('B', 'C')]

# Works right after the run, or later from the saved CSVs (e.g. after a Colab restart)
files = sorted(EVAL_DIR.glob('scenario_*/frame_metrics.csv'))
if 'all_frames' not in globals() and files:
    all_frames = pd.concat([pd.read_csv(p) for p in files], ignore_index=True)

if 'all_frames' not in globals():
    print('No main-experiment results yet: skipped.')
else:


    def compare(df):
        """Paired tests on frames where both methods have a value. Wilcoxon signed-rank for continuous metrics,
        exact McNemar (binomial test on discordant frames) for yes/no metrics."""
        key = ['scenario', 'frame_id']
        flow = df[df.method == 'C'].set_index(key).mean_flow_px
        rows = []
        for subset in ('all', 'moving'):
            for scen in sorted(df.scenario.unique()) + ['ALL']:
                part = df if scen == 'ALL' else df[df.scenario == scen]
                for metric in [m for m in CONTINUOUS + BINARY if m in df.columns]:
                    wide = part.pivot_table(index=key, columns='method', values=metric)
                    if subset == 'moving':
                        wide = wide[flow.reindex(wide.index) > MOVING_FLOW_PX]
                    for a, b in PAIRS:
                        w = wide[[a, b]].dropna()
                        if len(w) < 10:
                            continue
                        if metric in BINARY:
                            only_a = int(((w[a] == 1) & (w[b] == 0)).sum())
                            only_b = int(((w[b] == 1) & (w[a] == 0)).sum())
                            p = binomtest(only_b, only_a + only_b).pvalue if only_a + only_b else 1.0
                            note = f'only {a}: {only_a}, only {b}: {only_b}'
                        else:
                            p = wilcoxon(w[a], w[b]).pvalue if (w[a] != w[b]).any() else 1.0
                            note = ''
                        ma, mb = w[a].mean(), w[b].mean()
                        rows.append(dict(subset=subset, scenario=scen, metric=metric, comparison=f'{a} vs {b}', n=len(w),
                                         mean_first=ma, mean_second=mb,
                                         change_pct=100 * (mb - ma) / ma if ma else np.nan,
                                         p_value=p, significant=p < 0.05, note=note))
        return pd.DataFrame(rows)


    stats = compare(all_frames)
    stats.to_csv(EVAL_DIR / 'statistics.csv', index=False)
    print('Saved', EVAL_DIR / 'statistics.csv')
    with pd.option_context('display.max_rows', 300, 'display.width', 220):
        display(stats[stats.scenario == 'ALL'].round(4))

    # Per-scenario comparison of the main metrics
    metrics = [('temporal_deviation_px', 'Frame-to-frame deviation (px)'), ('goal_shift_px', 'Goal shift (px)'),
               ('violates_margin', 'Margin violation rate'), ('success', 'Success rate')]
    metrics = [(col, title) for col, title in metrics if col in all_frames.columns]
    names = sorted(all_frames.scenario.unique()) + ['ALL']
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 4))
    x = np.arange(len(names))
    for ax, (col, title) in zip(axes, metrics):
        for k, m in enumerate(METHODS):
            vals = [(all_frames if n == 'ALL' else all_frames[all_frames.scenario == n]).query('method == @m')[col].mean() for n in names]
            ax.bar(x + (k - 1) * 0.27, vals, 0.27, label=LABELS[m], color=np.array(METHOD_COLORS[m][::-1]) / 255)
        ax.set_xticks(x, names, rotation=30, ha='right'); ax.set_title(title); ax.grid(axis='y', alpha=0.25)
    axes[0].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(EVAL_DIR / 'comparison_by_scenario.png', dpi=150); plt.show()

## Cost-source comparison: semantic vs geometric vs combined

How much does each foundation model contribute? The same planner runs on six cost maps per frame:

| | Semantic only | Geometric only | Combined |
|---|---|---|---|
| **Single frame** (as Method A) | ✓ | ✓ | ✓ |
| **Temporal fusion** (as Method B) | ✓ | ✓ | ✓ |

Each map gets identical processing (hood removal, hard obstacles, resizing, inflation, endpoint selection). No RAFT, so this runs faster than the main experiment.

There is no ground-truth traversability, so each path is scored **against all three maps**. `crosses_geometric` for a semantic-only path, for example, is how often that path drives into an obstacle that only the depth model sees. This shows what each model misses and whether combining them helps. It does not prove which one is closer to reality.

Saves per scenario `sources/sources.mp4` (top row single frame, bottom row temporal fusion) and `sources/frame_metrics_sources.csv`.

In [ ]:
SOURCES = ('semantic', 'geometric', 'combined')
TEMPORAL = ('single', 'fusion')                     # single = A* on the current map, fusion = same as Method B
SOURCE_COLORS = {'semantic': (180, 70, 200), 'geometric': (200, 160, 40), 'combined': (60, 200, 255)}   # BGR
SOURCE_LABELS = {'semantic': 'Semantic only', 'geometric': 'Geometric only', 'combined': 'Combined'}
CONFIGS = [(s, t) for t in TEMPORAL for s in SOURCES]
SOURCES_MAX_FRAMES = MAX_FRAMES                     # set separately if you want a shorter source comparison


def source_cost(semantic, geometric, source):
    """Same processing as combine() (hood, resize, hard obstacles) but using one map only."""
    if source == 'combined':
        return combine(semantic, geometric)
    a = semantic if source == 'semantic' else geometric
    if HOOD_ROWS:
        a = a.copy()
        a[-HOOD_ROWS:] = np.minimum(a[-HOOD_ROWS - 1], 0.9 * OBSTACLE_THRESHOLD)
    small = cv2.resize(a, PLAN_SIZE, interpolation=cv2.INTER_AREA)
    if HARD_OBSTACLES:
        hard = cv2.resize((a >= OBSTACLE_THRESHOLD).astype(np.float32), PLAN_SIZE, interpolation=cv2.INTER_AREA) > 0
        small = np.where(hard, 1.0, small)
    return small.astype(np.float32)


def against(path, ref):
    """Score a path against one reference map: mean cost, obstacle crossing, margin violation, min clearance."""
    if path is None or len(path) < 2:
        return np.nan, np.nan, np.nan, np.nan
    h, w = ref['cost'].shape
    s = sample_path(path)
    r = np.clip(np.rint(s[:, 0]).astype(int), 0, h - 1)
    c = np.clip(np.rint(s[:, 1]).astype(int), 0, w - 1)
    clear = ref['clearance'][r, c]
    return (float(ref['cost'][r, c].mean()), float(ref['obstacles'][r, c].any()),
            float(clear.min() < CLEARANCE_PX - 1.0), float(clear.min()))


def source_panel(bgr, cost, path, title, colour, status):
    coloured = COST_LUT[np.clip(cost * 255, 0, 255).astype(np.uint8)]
    img = cv2.addWeighted(coloured, 0.35, bgr, 0.65, 0)
    img = draw_corridor(img, path, fill=colour, centre=(255, 255, 255))
    return no_path(label(img, title), status)


def run_sources(folder):
    """Plan with semantic-only, geometric-only and combined costs, each with and without temporal fusion."""
    print('\n' + folder.name + ' (cost-source comparison)')
    paths, indexes, ids = get_data(folder, verbose=False, limit=False)
    if SOURCES_MAX_FRAMES:
        ids = ids[:SOURCES_MAX_FRAMES]
    out = EVAL_DIR / folder.name / 'sources'
    out.mkdir(parents=True, exist_ok=True)
    zips = {k: zipfile.ZipFile(local_copy(p)) for k, p in paths.items()}
    set_hood(zips, indexes, ids)
    video, rows = None, []
    prev = dict(fid=None, hist={cfg: None for cfg in CONFIGS}, paths={cfg: None for cfg in CONFIGS},
                goals={cfg: None for cfg in CONFIGS})
    raw_video = CACHE / f'{folder.name}_sources.mp4'
    try:
        for fid in tqdm(ids, desc=folder.name):
            rgb, hw = read_rgb(zips['rgb'], indexes['rgb'][fid])
            sem = read_cost(zips['semantic'], indexes['semantic'][fid], hw)
            geo = read_cost(zips['geometric'], indexes['geometric'][fid], hw)
            if RESET_ON_GAP and prev['fid'] is not None and fid != prev['fid'] + 1:
                prev.update(hist={cfg: None for cfg in CONFIGS}, paths={cfg: None for cfg in CONFIGS},
                            goals={cfg: None for cfg in CONFIGS})
            current = {s: source_cost(sem, geo, s) for s in SOURCES}
            refs = {s: reference(current[s]) for s in SOURCES}
            bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
            panels, statuses = [], []
            for src, mode in CONFIGS:
                cfg = (src, mode)
                cost = current[src] if mode == 'single' else fuse(current[src], prev['hist'][cfg])
                t = time.perf_counter()
                path, status, used_goal = plan(cost, prev_goal=prev['goals'][cfg])
                plan_ms = 1000 * (time.perf_counter() - t)
                row = dict(scenario=folder.name, frame_id=fid, source=src, temporal=mode, config=f'{src}/{mode}',
                           success=int(path is not None), status=status, plan_ms=plan_ms,
                           length_px=float(np.linalg.norm(np.diff(path, axis=0), axis=1).sum()) if path is not None else np.nan,
                           temporal_deviation_px=temporal_deviation(prev['paths'][cfg], path, PLAN_SIZE[1]),
                           goal_shift_px=float(np.linalg.norm(path[-1] - prev['paths'][cfg][-1]))
                           if path is not None and prev['paths'][cfg] is not None else np.nan)
                for ref_name in SOURCES:
                    cost_r, cross_r, margin_r, clear_r = against(path, refs[ref_name])
                    row.update({f'cost_on_{ref_name}': cost_r, f'crosses_{ref_name}': cross_r,
                                f'margin_{ref_name}': margin_r, f'min_clearance_{ref_name}': clear_r})
                rows.append(row)
                prev['paths'][cfg] = path
                if path is not None:
                    prev['goals'][cfg] = used_goal
                prev['hist'][cfg] = cost if FUSION_HISTORY == 'fused' else current[src]
                statuses.append(status)
                title = f'{SOURCE_LABELS[src]}  |  ' + ('single frame' if mode == 'single' else 'temporal fusion')
                panels.append(source_panel(bgr, cost, path, title, SOURCE_COLORS[src], status))
            footer = f'{folder.name}   frame {fid}   |   top: single-frame A*   bottom: temporal fusion + A*   |   ' \
                     + '  '.join(f'{s[:3]}/{m[:3]}: {st}' for (s, m), st in zip(CONFIGS, statuses))
            frame = grid(panels, 3, footer)
            if video is None:
                video = cv2.VideoWriter(str(raw_video), cv2.VideoWriter_fourcc(*'mp4v'), FPS, (frame.shape[1], frame.shape[0]))
            video.write(frame)
            prev['fid'] = fid
    finally:
        if video is not None:
            video.release()
        for z in zips.values():
            z.close()
    result = subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw_video), '-vcodec', 'libx264',
                             '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(out / 'sources.mp4')])
    if result.returncode != 0:
        shutil.copy(raw_video, out / 'sources.mp4')
    df = pd.DataFrame(rows)
    df.to_csv(out / 'frame_metrics_sources.csv', index=False)
    print('Saved to', out)
    return df


if RUN_SOURCE_EXPERIMENT:
    source_frames = pd.concat([run_sources(folder) for folder in scenarios], ignore_index=True)


### Cost-source results

The crossing table is the key result: rows = map used for planning, columns = map used for checking. Off-diagonal values show what a single model misses. Tests: Wilcoxon for stability, exact McNemar for crossings and margin violations. Saves `sources_summary.csv`, `sources_statistics.csv` and `sources_comparison.png`. Can be rerun later from the saved CSVs.

In [ ]:
from scipy.stats import wilcoxon, binomtest

files = sorted(EVAL_DIR.glob('scenario_*/sources/frame_metrics_sources.csv'))
if 'source_frames' not in globals() and files:
    source_frames = pd.concat([pd.read_csv(p) for p in files], ignore_index=True)

if 'source_frames' not in globals():
    print('No cost-source results yet: skipped.')
else:


    def summarize_sources(df):
        key = ['scenario', 'frame_id']
        wide = df.pivot_table(index=key, columns='config', values='temporal_deviation_px')
        common = wide.dropna().index
        df = df.assign(_common=df.set_index(key).index.isin(common))
        rows = []
        for (src, mode), part in df.groupby(['source', 'temporal']):
            ok = part[part.success == 1]
            row = dict(source=src, temporal=mode, frames=len(part), success_rate=part.success.mean(),
                       temporal_dev_common_px=part[part._common].temporal_deviation_px.mean(),
                       mean_goal_shift_px=part.goal_shift_px.mean(), mean_length_px=ok.length_px.mean(),
                       mean_plan_ms=part.plan_ms.mean())
            for ref in SOURCES:
                row[f'crosses_{ref}'] = ok[f'crosses_{ref}'].mean()
                row[f'margin_{ref}'] = ok[f'margin_{ref}'].mean()
                row[f'cost_on_{ref}'] = ok[f'cost_on_{ref}'].mean()
            rows.append(row)
        order = {s: i for i, s in enumerate(SOURCES)}
        return pd.DataFrame(rows).sort_values(['temporal', 'source'], key=lambda c: c.map(order) if c.name == 'source' else c)


    def compare_sources(df):
        """Paired tests: combined vs each single source (same temporal mode), and fusion vs single (same source)."""
        key = ['scenario', 'frame_id']
        pairs = [((a, m), ('combined', m)) for m in TEMPORAL for a in ('semantic', 'geometric')]
        pairs += [((s, 'single'), (s, 'fusion')) for s in SOURCES]
        rows = []
        for metric in ['temporal_deviation_px', 'goal_shift_px'] + [f'crosses_{r}' for r in SOURCES] + [f'margin_{r}' for r in SOURCES]:
            wide = df.pivot_table(index=key, columns='config', values=metric)
            for (s1, m1), (s2, m2) in pairs:
                c1, c2 = f'{s1}/{m1}', f'{s2}/{m2}'
                w = wide[[c1, c2]].dropna()
                if len(w) < 10:
                    continue
                if metric.startswith(('crosses', 'margin')):
                    only1 = int(((w[c1] == 1) & (w[c2] == 0)).sum()); only2 = int(((w[c2] == 1) & (w[c1] == 0)).sum())
                    p = binomtest(only2, only1 + only2).pvalue if only1 + only2 else 1.0
                    note = f'only {c1}: {only1}, only {c2}: {only2}'
                else:
                    p = wilcoxon(w[c1], w[c2]).pvalue if (w[c1] != w[c2]).any() else 1.0
                    note = ''
                rows.append(dict(metric=metric, comparison=f'{c1} vs {c2}', n=len(w), mean_first=w[c1].mean(),
                                 mean_second=w[c2].mean(), p_value=p, significant=p < 0.05, note=note))
        return pd.DataFrame(rows)


    src_summary = pd.concat([summarize_sources(part).assign(scenario=name) for name, part in source_frames.groupby('scenario')]
                            + [summarize_sources(source_frames).assign(scenario='ALL')], ignore_index=True)
    src_summary = src_summary[['scenario'] + [c for c in src_summary.columns if c != 'scenario']]
    src_summary.to_csv(EVAL_DIR / 'sources_summary.csv', index=False)
    src_stats = compare_sources(source_frames)
    src_stats.to_csv(EVAL_DIR / 'sources_statistics.csv', index=False)

    pooled = src_summary[src_summary.scenario == 'ALL'].set_index(['temporal', 'source'])
    print('Obstacle crossings: rows = map used for planning, columns = map used for checking (all scenarios)')
    display((100 * pooled[[f'crosses_{r}' for r in SOURCES]]).round(2).rename(columns=lambda c: c.replace('crosses_', 'checked on ')))
    print('Margin violations (%):')
    display((100 * pooled[[f'margin_{r}' for r in SOURCES]]).round(2).rename(columns=lambda c: c.replace('margin_', 'checked on ')))
    with pd.option_context('display.max_rows', 200, 'display.width', 220):
        display(pooled[['success_rate', 'temporal_dev_common_px', 'mean_goal_shift_px', 'mean_length_px', 'mean_plan_ms']].round(3))
        display(src_stats.round(4))

    # Figure: stability and safety per configuration
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    x = np.arange(len(TEMPORAL))
    for k, src in enumerate(SOURCES):
        colour = np.array(SOURCE_COLORS[src][::-1]) / 255
        sub = pooled.xs(src, level='source').reindex(list(TEMPORAL))
        axes[0].bar(x + (k - 1) * 0.27, sub.temporal_dev_common_px, 0.27, color=colour, label=SOURCE_LABELS[src])
        axes[1].bar(x + (k - 1) * 0.27, sub.success_rate, 0.27, color=colour)
        axes[2].bar(x + (k - 1) * 0.27, 100 * sub.crosses_geometric, 0.27, color=colour)
        axes[3].bar(x + (k - 1) * 0.27, 100 * sub.crosses_semantic, 0.27, color=colour)
    for ax, title in zip(axes, ['Frame-to-frame deviation (px)', 'Success rate', 'Crosses geometric obstacles (%)',
                                'Crosses semantic obstacles (%)']):
        ax.set_xticks(x, ['single frame', 'temporal fusion']); ax.set_title(title); ax.grid(axis='y', alpha=0.25)
    axes[0].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(EVAL_DIR / 'sources_comparison.png', dpi=150); plt.show()
    print('Saved', EVAL_DIR / 'sources_summary.csv', EVAL_DIR / 'sources_statistics.csv', EVAL_DIR / 'sources_comparison.png')


## Interpreting the results

- All metrics are **image-space proxies** on predicted cost maps. No ground-truth traversability is available, so success, cost and clearance do not prove the vehicle could physically drive the path.
- `margin_violation_rate`: share of paths that come more than 1 px inside the 12 px safety margin of obstacles in the current frame. `goal_shift_px`: how far the goal moved since the previous frame (the main source of path jumps).
- `crosses_obstacle` / `margin_violation_rate` measure how often a B or C path (planned on a fused map) passes through obstacles **visible in the current frame**. Fusion can average away a newly appeared obstacle; this is the main safety cost of temporal smoothing. `PRESERVE_CURRENT_OBSTACLES = True` removes that risk and is a useful ablation.
- `temporal_dev_common_px` compares methods only on transitions where all three planned successfully; use it rather than `temporal_dev_all_px` for the stability comparison. It is lateral deviation in image pixels between consecutive paths, not real-world trajectory error, and part of it is genuine ego-motion.
- `mean_abs_B_C` near zero means RAFT alignment had no effect (little motion, or the warp was mostly rejected — check `warp_valid_frac`). RAFT is not expected to win by default.
- `GOAL_STICKINESS = 0` (default) keeps Method A a pure single-frame baseline. A value such as 1.0 makes all three methods prefer a goal near the previous one: a separate ablation, not part of the main comparison.
- Use `RUN_TAG` to keep ablation runs (e.g. `FUSION_ALPHA = 0.4`, `PRESERVE_CURRENT_OBSTACLES = True`) in separate folders.
- With `SHARED_ENDPOINTS = False` (default) each method chooses its goal on its own map, so fusion can also steady the goal. With `True`, all methods share the goal chosen on the current map; the `status` column records why each failure happened.
- The drawn corridor (`VEHICLE_WIDTH_FRAC`, `HORIZON_FRAC`) is for display only and approximates the vehicle width; the planner's 12 px margin does not scale with distance.